In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*Viz [referenci API](https://docs.quantum.ibm.com/api/functions/kipu-optimization)*

> **Note:** * Funkce Qiskit jsou experimentální funkce dostupné pouze uživatelům plánů IBM Quantum&reg; Premium, Flex a On-Prem (prostřednictvím rozhraní IBM Quantum Platform API). Jsou ve stavu preview release a mohou se změnit.

## Přehled

S Iskay Quantum Optimizerem od Kipu Quantum můžeš řešit složité optimalizační úlohy na kvantových počítačích IBM&reg;. Tento solver využívá špičkový [algoritmus bf-DCQO](https://doi.org/10.48550/arXiv.2409.04477) od Kipu, který jako vstup vyžaduje pouze účelovou funkci a automaticky poskytuje řešení daného problému. Dokáže zpracovat optimalizační úlohy zahrnující až 156 qubitů, čímž umožňuje využít všechny qubity kvantových zařízení IBM. Optimizer používá mapování 1:1 mezi klasickými proměnnými a qubity, což ti umožňuje řešit optimalizační úlohy s až 156 binárními proměnnými.

Optimizer umožňuje řešení neomezených binárních optimalizačních problémů. Kromě běžně používané formulace QUBO (Quadratic Unconstrained Binary Optimization) podporuje také optimalizační problémy vyššího řádu (HUBO). Solver využívá nevariační kvantový algoritmus a většinu výpočtů provádí na kvantových zařízeních.

Níže najdeš podrobnější informace o použitém algoritmu a stručný návod k použití funkce, spolu s výsledky benchmarkování na různých instancích problémů různých velikostí a složitostí.

## Popis

Optimizer je připravená implementace špičkových kvantových optimalizačních algoritmů. Řeší optimalizační problémy spouštěním vysoce komprimovaných kvantových obvodů na kvantovém hardware. Tato komprese je dosažena zavedením kontradiabatických členů do základní časové evoluce kvantového systému. Algoritmus provádí několik iterací běhů na hardware, aby získal konečná řešení, a kombinuje je s následným zpracováním. Tyto kroky jsou plynule integrovány do pracovního postupu Optimizeru a jsou prováděny automaticky.

### Jak Quantum Optimizer funguje?

Tato část popisuje základy implementovaného algoritmu bf-DCQO. Úvod do algoritmu najdeš také na [kanálu Qiskit na YouTube](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s).

Algoritmus je založen na časové evoluci kvantového systému, který se v čase transformuje, přičemž řešení problému je zakódováno v základním stavu kvantového systému na konci evoluce. Podle [adiabatického teorému](https://en.wikipedia.org/wiki/Adiabatic_theorem) musí být tato evoluce pomalá, aby se zajistilo, že systém zůstane ve svém základním stavu. Digitalizace této evoluce je základem digitalizovaného kvantového adiabatického výpočtu (DQA) a nechvalně známého algoritmu QAOA. Pomalá evoluce však není pro rostoucí velikosti problémů proveditelná, protože vede k rostoucí hloubce obvodu. Pomocí kontradiabatických protokolů lze potlačit nežádoucí excitace vznikající při krátkých dobách evoluce a přitom zůstat v základním stavu. Digitalizace této kratší doby evoluce zde vede ke kvantovým obvodům s menší hloubkou a méně zapletovacími hradly.

Obvody algoritmů bf-DCQO obvykle využívají až desetkrát méně zapletovacích hradel než DQA a třikrát až čtyřikrát méně zapletovacích hradel než standardní implementace QAOA. Díky menšímu počtu hradel dochází při provádění obvodů na hardware k méně chybám. Optimizer proto nevyžaduje použití technik jako potlačení nebo zmírnění chyb. Jejich implementace v budoucích verzích může dále zlepšit kvalitu řešení.

Ačkoli algoritmus bf-DCQO používá iterace, je nevariační. Po každé iteraci algoritmu je měřeno rozdělení stavů. Získané rozdělení se používá k výpočtu tzv. pole předpojatosti (bias-field). Toto pole umožňuje zahájit další iteraci z energetického stavu blízkého dříve nalezenému řešení. Tímto způsobem algoritmus s každou iterací postupuje k řešením nižší energie. Typicky postačuje přibližně deset iterací ke konvergenci k řešení, celkem tedy vyžaduje podstatně nižší počet iterací než variační algoritmy, jejichž počet se pohybuje v řádu přibližně 100 iterací.

Optimizer kombinuje algoritmus bf-DCQO s klasickým následným zpracováním. Po naměření rozdělení stavů se provede lokální prohledávání. Během lokálního prohledávání jsou bity naměřeného řešení náhodně přehazovány. Po přehození je vyhodnocena energie nového bitového řetězce. Pokud je energie nižší, je bitový řetězec ponechán jako nové řešení. Lokální prohledávání roste pouze lineárně s počtem qubitů, takže je výpočetně nenáročné. Protože následné zpracování opravuje lokální bitové přehody, kompenzuje chyby bitových přechodů, které jsou často důsledkem nedokonalosti hardware a chyb při čtení.

### Pracovní postup

Níže je schéma pracovního postupu Quantum Optimizeru.

![Pracovní postup](../docs/images/guides/kipu-optimization/workflow.svg "Pracovní postup Quantum Optimizeru")

Použitím Quantum Optimizeru lze řešení optimalizačního problému na kvantovém hardware zredukovat na:
* Formulaci účelové funkce problému
* Přístup k Optimizeru přes Qiskit Functions
* Spuštění Optimizeru a sběr výsledků

## Benchmarky

Níže uvedené metriky benchmarků ukazují, že Optimizer efektivně řeší problémy zahrnující až 156 qubitů a nabízí obecný přehled přesnosti a škálovatelnosti optimizeru napříč různými typy problémů. Skutečné výkonnostní metriky se mohou lišit v závislosti na konkrétních vlastnostech problému, jako je počet proměnných, hustota a lokalita členů v účelové funkci a polynomiální řád.

Následující tabulka obsahuje aproximační poměr (AR), metriku definovanou takto:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
kde $C$ je účelová funkce, $C_{\textrm{min}}$, $C_{\textrm{max}}$ jsou její minimální a maximální hodnoty a $C^{*}$ je cena nejlepšího nalezeného řešení. AR=100\% tedy znamená, že byl nalezen základní stav problému.

| Příklad           | Počet qubitů | Aproximační poměr | Celkový čas (s) | Využití runtime (s) | Celkový počet shotů | Počet iterací |
| ----------------- | :----------: | :---------------: | :-------------: | :-----------------: | :-----------------: | :-----------: |
| Unweighted MaxCut |      28      |       100%        |       180       |         30          |         30k         |       5       |
| Unweighted MaxCut |      30      |       100%        |       180       |         30          |         30k         |       5       |
| Unweighted MaxCut |      32      |       100%        |       180       |         30          |         30k         |       5       |
| Unweighted MaxCut |      80      |       100%        |       480       |         60          |         90k         |       9       |
| Unweighted MaxCut |     100      |       100%        |       330       |         60          |         60k         |       6       |
| Unweighted MaxCut |     120      |       100%        |       370       |         60          |         60k         |       6       |
| HUBO 1            |     156      |       100%        |       600       |         70          |        100k         |      10       |
| HUBO 2            |     156      |       100%        |       600       |         70          |        100k         |      10       |

- Instance MaxCut s 28, 30 a 32 qubity byly spuštěny na ibm_sherbrooke. Instance s 80, 100 a 120 byly spuštěny na procesoru Heron r2.
- Instance HUBO byly také spuštěny na procesoru Heron r2.

Všechny instance benchmarků jsou dostupné na GitHubu (viz [instance benchmarků Kipu](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Příklad ke spuštění těchto instancí najdeš v části [Příklad 3: Instance benchmarků](#example-3-benchmark-instances).

## Začínáme

V této dokumentaci projdeme kroky pro použití nástroje Iskay Quantum Optimizer. Ukážeme si, jak načíst funkci z katalogu a jak převést svůj problém na platný vstup, přičemž si zároveň ukážeme, jak můžeš experimentovat s různými volitelnými parametry.

Podrobnější příklad najdeš v tutoriálu [Řešení problému Market Split s nástrojem Iskay Quantum Optimizer od Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), kde procházíme celý proces použití Iskay Solveru k řešení problému Market Split – reálné výzvy v oblasti alokace zdrojů, kde je třeba rozdělit trhy do vyvážených prodejních regionů tak, aby byly splněny přesné cíle poptávky.

Ověř svoji totožnost pomocí API klíče, který najdeš na [řídicím panelu IBM Quantum Platform](http://quantum.cloud.ibm.com/), a vyber Qiskit Function následovně:

In [ ]:
# ruff: noqa: F821

> **Note:** Následující kód předpokládá, že máš uložené přihlašovací údaje. Pokud ne, postupuj podle pokynů v části [uložení účtu IBM Cloud](/guides/functions#install-qiskit-functions-catalog-client) a ověř se svým API klíčem.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Vlastní příklad konfigurace
Ukázka, jak lze nakonfigurovat Iskay s různými nastaveními:

In [ ]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Optimalizace Seed**: Mějte na paměti, že `seed_transpiler` je ve výchozím nastavení nastaven na `None`. To umožňuje automatický optimalizační proces transpileru. Je-li hodnota `None`, systém spustí pokus s více semeny a vybere to, které vytváří nejlepší hloubku obvodu, čímž plně využívá parametr `max_trials` pro každou úroveň transpilace.

**Výkon úrovně transpilace**: Zvýšení počtu `max_trials` s vyššími hodnotami pro `transpilation_level` nevyhnutelně zvýší čas transpilace, ale nemusí vždy změnit konečný obvod – to velmi závisí na konkrétní struktuře a složitosti obvodu. Pro některé obvody/problémy však může být rozdíl mezi 10 pokusy (úroveň 1) a 50 pokusy (úroveň 5) dramatický, takže prozkoumání těchto parametrů by mohlo být klíčem k úspěšnému nalezení řešení.

## Příklad 1: Jednoduchá účelová funkce

Uvažujme účelovou funkci ve spinové formulaci:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

kde $(x_0, ..., x_4) \in {-1, 1}^5$.

Řešení této jednoduché účelové funkce je
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
s minimální hodnotou $C^{*} = -6$

### 1. Vytvoření účelové funkce

Začneme vytvořením slovníku s koeficienty účelové funkce:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Spuštění optimalizátoru
Problém vyřešíme spuštěním optimalizátoru. Protože $(x_0, ..., x_4) \in {-1, 1}^5$, musíme nastavit `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Získání výsledku
Řešení optimalizačního problému je poskytnuto přímo z optimalizátoru.

In [ ]:
print(job.result())

Zobrazí se slovník ve tvaru:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Všimni si, že slovník `solution` zobrazuje vektor výsledku $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## Příklad 2: MaxCut

Mnohé grafové problémy, jako je MaxCut nebo Maximum independent set, jsou NP-těžké problémy a ideální kandidáti pro testování kvantových algoritmů a hardwaru. Tento příklad demonstruje řešení problému MaxCut na 3-regulárním grafu pomocí Quantum Optimizer.

Aby bylo možné tento příklad spustit, musíš kromě `qiskit-ibm-catalog` nainstalovat také balíček `networkx`. Pro instalaci spusť následující příkaz:

In [ ]:
# %pip install networkx numpy

### 1. Vytvoření účelové funkce
Začni vygenerováním náhodného 3-regulárního grafu. Pro tento graf definujeme účelovou funkci problému MaxCut.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the max-cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Spuštění Optimizeru
Vyřeš problém spuštěním optimizeru.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Získání výsledku
Získej výsledek a namapuj bitový řetězec řešení zpět na původní uzly grafu.

In [ ]:
print(job.result())

Řešení problému MaxCut je přímo obsaženo v sub-slovníku `solution` výsledného objektu.

In [ ]:
maxcut_solution = job.result()["solution"]

## Příklad 3: Benchmarkové instance
Benchmarkové instance jsou dostupné na GitHubu: [Kipu benchmark instances](https://github.com/Kipu-Quantum-GmbH/benchmark-instances).

Instance lze načíst pomocí knihovny `pygithub`. Pro instalaci spusť následující příkaz:

In [ ]:
# %pip install pygithub

Cesty k benchmarkovým instancím jsou:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Pro reprodukci výkonu benchmarku pro HUBO instance zvol backend `ibm_marrakesh` a nastav `direct_qubit_mapping` na `True` v sub-slovníku `options`.
Následující příklad spustí instanci Maxcut s 32 uzly.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_regular_3_150_nodes_weighted.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "<BACKEND-NAME>",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Případy použití
Typickými případy použití pro Optimization solver jsou problémy kombinatorické optimalizace. Problémy lze řešit z mnoha odvětví, jako jsou finance, farmaceutický průmysl nebo logistika. Několik příkladů:
* Optimalizace portfolia (QUBO): [vědecká publikace](https://doi.org/10.1103/PhysRevApplied.22.054037) a [white paper](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Skládání proteinů (HUBO): [vědecká publikace](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Plánování v logistice (QUBO): [vědecká publikace](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Optimalizace sítí: [webinář](https://www.youtube.com/watch?v=w5SrCIK88No)
* Rozdělení trhu (QUBO): [tutoriál](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Pokud máš zájem řešit konkrétní případ použití a vyvinout pro něj dedikované mapování, můžeme ti pomoci. [Kontaktuj nás](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5).
## Získání podpory
Pro podporu kontaktuj support@kipu-quantum.com.
## Další kroky
- [Požádej o přístup k Quantum Optimizer od Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5).
- Navštiv [referenci API](https://docs.quantum.ibm.com/api/functions/kipu-optimization) pro tuto funkci Qiskit.
- Vyzkoušej tutoriál [Řešení problému rozdělení trhu s Iskay Quantum Optimizer od Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- Prostuduj [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477](https://arxiv.org/abs/2409.04477).
- Prostuduj [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898](https://arxiv.org/abs/2405.13898).
- Prostuduj [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663](https://arxiv.org/abs/2505.08663).
## Další informace
Iskay, podobně jako název naší společnosti Kipu Quantum, je peruánské slovo. Přestože jsme startup z Německa, tato slova pocházejí z rodné země jednoho z našich spoluzakladatelů, kde byla Quipu jedním z prvních výpočetních strojů vytvořených lidstvem před 2000 lety př. n. l.